<a href="https://colab.research.google.com/github/Alenushka2013/ML_for_people_tasks/blob/main/HW_6_Using_prompts_and_agents_in_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [1]:
!pip -q install langchain langchain_openai huggingface_hub openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 1.8 MB/s eta 0:00:00


In [22]:
from huggingface_hub import InferenceClient
import json

with open("creds.json") as file:
    creds = json.load(file)

client = InferenceClient(
    "mistralai/Mistral-7B-Instruct-v0.2",
    token=creds["HUGGINGFACEHUB_API_TOKEN"]
)

prompt = """
Тема: "Квантові обчислення".
Дай визначення, ключові переваги та поточні дослідження.
Відповідь простою мовою, обмеж до 200 символів, будь лаконічним.
"""

response = client.chat_completion(
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
    temperature=0
)

print(response.choices[0].message["content"])


 Квантове обчислення - це область комп'ютерних наук, яка використовує квантову механіку для розв'язання складних математичних задач швидше, ніж класичні комп'ютери. Ключові переваги: паралельність обчислень (квабіт може бути у двох станах одночасно), висока швидкість для специфічних задач (наприклад, шифрування та оптимізація). Поточні дослідження: розвиток квантових алгоритмів, створення надійніших квантових бітів, покращення квантових комп'ютерів.


Температура рівна 0 відповідає мінімальній креативності мовної моделі. Визначення наукового терміну хотілося б отримати точним, без зайвих вигадок.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [23]:
from langchain.prompts import PromptTemplate

# --- Параметризований промпт через LangChain ---
template = """
Тема: "{topic}".
Дай визначення, ключові переваги та поточні дослідження.
Відповідь простою мовою, обмеж до 200 символів, будь лаконічним.
"""
prompt = PromptTemplate(template=template, input_variables=["topic"])

# --- Список тем ---
topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI"
]

# --- Генерація відповідей ---
for t in topics:
    formatted_prompt = prompt.format(topic=t)

    response = client.chat_completion(
        messages=[{"role": "user", "content": formatted_prompt}],
        max_tokens=200,
        temperature=0.1
    )

    print(f"Тема: {t}")
    print(response.choices[0].message["content"])
    print("-" * 50)

Тема: Баєсівські методи в машинному навчанні
 Баєсівські методи - форма машинного навчання, де використовується теорія імовірностей для визначення ймовірності класу на основі прийнятих припущень. Вони дозволяють оновлювати знання за новою інформацією, маючи м'яку початкову модель. Ключові переваги: гнучкість, здатність до інтерпретації, обробка неповних даних. Поточні дослідження: масштабованість, швидкість, інтеграція з іншими методами.
--------------------------------------------------
Тема: Трансформери в машинному навчанні
 Transformers in Machine Learning:

Transformers are a type of neural network model introduced by Google in 2017, revolutionizing natural language processing (NLP) with their ability to handle long-range dependencies. In ML, they're used for tasks like text generation, translation, and summarization.

Key benefits:
1. Long-range dependency handling: Transformers can process long sequences effectively, unlike RNNs.
2. Parallel processing: Transformers use self-att



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [9]:
!pip install -q langchain_community duckduckgo_search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

"2 of 2. Barack Obama: timeline Key events in the life of Barack Obama. Barack Obama (born August 4, 1961, Honolulu, Hawaii, U.S.) is the 44th president of the United States (2009-17) and the first African American to hold the office. Before winning the presidency, Obama represented Illinois in the U.S. Senate (2005-08). Since the office was established in 1789, 45 men have served in 46 presidencies. The first president, George Washington, won a unanimous vote of the Electoral College. [4] Grover Cleveland served two non-consecutive terms and is therefore counted as the 22nd and 24th president of the United States, giving rise to the discrepancy between the ... Here is a list of the presidents and vice presidents of the United States along with their parties and dates in office. ... Chester A Arthur: Twenty-First President of the United States. 10 Interesting Facts About James Buchanan. Martin Van Buren - Eighth President of the United States. Quotes From Harry S. Truman. Table of Cont



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?
